In [ ]:
print("Hello")

In [ ]:
##

## 데이터 준비
 -  다운받기,  내가 직접 모으기
 -  다듬기( 정제, 전처리 )
 -  정규화  255.0
 - train, val, test

## 모델만들기
Conv , pool,  FC 연결

## 학습하기 (검증 세트 가능)
compile( )
fit(...., callback, validation_data=( , ) )

## 평가하기 (예측, accuracy)
pred == y_true,
evaluate

## 성능 개선
1. 용량 늘이기
2. 데이터량 늘이기 augmentation (증강)
   -  회전, 반전 flip , crop, 확대, 축소..
3. Dropout , 규제, ....., batch-normalization , Early Stopping

In [ ]:
import keras
from keras.utils import image_dataset_from_directory
from keras import layers
from keras import models
import os

In [4]:
!gdown https://drive.google.com/uc?id=133qG35aMUd_u273w32aOO_TJfyO1CiOd

Downloading...
From: https://drive.google.com/uc?id=133qG35aMUd_u273w32aOO_TJfyO1CiOd
To: /content/cats_and_dogs.tar
100% 94.0M/94.0M [00:01<00:00, 76.6MB/s]


In [5]:
!ls

cats_and_dogs.tar  sample_data


In [6]:
!tar -xf  cats_and_dogs.tar

In [7]:
!pwd

/content


In [8]:
train_dir = "./cats_and_dogs/train"
validation_dir = "./cats_and_dogs/validation"
test_dir = "./cats_and_dogs/test"

In [13]:
# 데이터셋 공급하기
from keras.utils import image_dataset_from_directory

train_dataset = image_dataset_from_directory(
    train_dir,
    image_size=(180, 180),  # image_size는 정해져 있지 않고 자유로움
    batch_size=16)
validation_dataset = image_dataset_from_directory(
    validation_dir,
    image_size=(180, 180),
    batch_size=16)
test_dataset = image_dataset_from_directory(
    test_dir,
    image_size=(180, 180),
    batch_size=16)


Found 2000 files belonging to 2 classes.
Found 1000 files belonging to 2 classes.
Found 1000 files belonging to 2 classes.


## 모델 만들기

In [18]:
inputs = layers.Input(shape=(180, 180, 3))
x = layers.Conv2D(filters=32, kernel_size=(3, 3), activation='relu')(inputs)
x = layers.MaxPooling2D(pool_size=(2, 2))(x)
x = layers.Conv2D(filters=64, kernel_size=(3, 3), activation='relu')(x)
x = layers.MaxPooling2D(pool_size=(2, 2))(x)
x = layers.Conv2D(filters=128, kernel_size=(3, 3), activation='relu')(x)
x = layers.MaxPooling2D(pool_size=(2, 2))(x)
x = layers.Conv2D(filters=256, kernel_size=(3, 3), activation='relu')(x)
x = layers.MaxPooling2D(pool_size=(2, 2))(x)
x = layers.Flatten()(x)
x = layers.Dense(units=256, activation='relu')(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(units=1, activation='sigmoid')(x)

model = keras.Model(inputs=inputs, outputs=outputs)


In [19]:
model.compile(loss='binary_crossentropy',
              optimizer='rmsprop',
              metrics=['accuracy'])

In [ ]:
model.fit(train_dataset, epochs=30, validation_data=validation_dataset )

In [22]:
model.evaluate(test_dataset)

63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.6950 - loss: 3.2647


[3.264744520187378, 0.6949999928474426]

In [23]:
model.save("convnet_from_scratch.keras")

In [24]:
mine = keras.models.load_model("convnet_from_scratch.keras")

In [25]:
mine.evaluate(test_dataset)

63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.6950 - loss: 3.2647


[3.264744520187378, 0.6949999928474426]

## 데이터 증강하기

In [27]:
data_augmentation = keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.2),
])

inputs = keras.Input(shape=(180, 180, 3))
x = data_augmentation(inputs)

x = layers.Rescaling(1./255)(x)

x = layers.Conv2D(filters=32, kernel_size=3, activation="relu")(x)
x = layers.MaxPooling2D(pool_size=2)(x)

x = layers.Conv2D(filters=64, kernel_size=3, activation="relu")(x)
x = layers.MaxPooling2D(pool_size=2)(x)

x = layers.Conv2D(filters=128, kernel_size=3, activation="relu")(x)
x = layers.MaxPooling2D(pool_size=2)(x)

x = layers.Conv2D(filters=256, kernel_size=3, activation="relu")(x)
x = layers.MaxPooling2D(pool_size=2)(x)

x = layers.Conv2D(filters=256, kernel_size=3, activation="relu")(x)
# 여기에 MaxPooling2D 가 있어도 된다

x = layers.Flatten()(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs=inputs, outputs=outputs)



In [ ]:
model.compile(loss='binary_crossentropy',
              optimizer='rmsprop',
              metrics=['accuracy'])
model.fit(train_dataset, epochs=30, validation_data=validation_dataset )

Epoch 1/30
125/125 ━━━━━━━━━━━━━━━━━━━━ 10s 38ms/step - accuracy: 0.5045 - loss: 0.6981 - val_accuracy: 0.5000 - val_loss: 0.6941
Epoch 2/30
125/125 ━━━━━━━━━━━━━━━━━━━━ 4s 32ms/step - accuracy: 0.5140 - loss: 0.6961 - val_accuracy: 0.5000 - val_loss: 0.7104
Epoch 3/30
125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 33ms/step - accuracy: 0.5355 - loss: 0.6918 - val_accuracy: 0.5140 - val_loss: 0.6854
Epoch 4/30
125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 30ms/step - accuracy: 0.5845 - loss: 0.6744 - val_accuracy: 0.6250 - val_loss: 0.6371
Epoch 5/30
125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 38ms/step - accuracy: 0.6315 - loss: 0.6463 - val_accuracy: 0.6150 - val_loss: 0.6514
Epoch 6/30
125/125 ━━━━━━━━━━━━━━━━━━━━ 4s 31ms/step - accuracy: 0.6485 - loss: 0.6422 - val_accuracy: 0.6560 - val_loss: 0.6159
Epoch 7/30
125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 30ms/step - accuracy: 0.6700 - loss: 0.6238 - val_accuracy: 0.6530 - val_loss: 0.6236
Epoch 8/30
125/125 ━━━━━━━━━━━━━━━━━━━━ 6s 37ms/step - accuracy: 0.6740 - loss: 0.6112 - val_acc